# **Sign language translator**
*Introduction to Deep Learning for Natural Language Processing - Final Project*  
**Author:** Wojciech Borysewicz  

### **Description**  
This file contains the full implementation of the model (code needed to reproduce all experiments). This Colab Notebook will be joined with an additional PDF file containing the report summarising the experiments conducted in this file.  

# **Code running and instalation**   

This part contains the code importing the necessary libraries and showing their versions, as well as the code chunk with frequently used constants.

### **Libraries**
  
---

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import pandas as pd
import os
import random
import math

# Note:
"""
Uncomment the next four lines only to check the libraries versions. Make sure
those lines are commented back before executing the project code. Those lines were
added after writing and executing the project code, so leaving the four "import ..."
commands may result in problems with function names.
"""

# import sys # to check the Python version
# import sklearn
# import IPython
# import matplotlib

from sklearn.model_selection import train_test_split
from pathlib import Path
from itertools import product
from IPython.display import display

In [ ]:
print(f"Versions:\n Python: {sys.version.split()[0]}, \n Matplotlib: {matplotlib.__version__}, \n Numpy: {np.__version__},\n Tensorflow: {tf.__version__},\n Pandas: {pd.__version__}, \n SciKitLearn: {sklearn.__version__} \n IPython: {IPython.__version__}")


### **Reproducibility and configuration**
  
---

In [ ]:
# Randomstate
SEEDS = [0, 1, 123, 2026, 12345]
SEED = 42 # using the popular "42" value
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Split setting
TEST_SIZE = 0.3
VAL_SPLIT = 0.1 # validation split
TEST_SPLITS = [0.2, 0.3, 0.5, 0.7] # avoiding the usual 80/20 split (due to dataset size)

# Model training
BATCH_SIZE = 32
EPOCHS = 25
SEARCH_EPOCHS = 8
LR = 0.001

# Dataset
IMG_SIZE = (64, 64)

# Final runs
FINAL_RUNS = 5

# **Dataset**

### **Images Dataset Class**
  
---

In [ ]:
# Helper function
def label_sort_key(label):
    """
    Returns a sorting key, in a form of a tuple, for dataset class labels.
    Numeric labels are sorted first, alphabetic labels - afterwards.
    """

    label = str(label)

    if label.isdigit():
      return (0, int(label))

    return (1, label.upper())

# The Images_Dataset class
class Images_Dataset:
    def __init__(self, folder):
        if isinstance(folder, str):
            self.folders = [folder]
        else:
            self.folders = folder

        self.categories = self.get_categories()
        self.images = self.load_images()

    def get_categories(self):
        categories = set()

        for folder in self.folders:
            for category in os.listdir(folder):
                category_path = os.path.join(folder, category)

                if os.path.isdir(category_path):
                    categories.add(category)

        return sorted(categories, key=label_sort_key)

    def load_images(self):
        """
        Loads all image files from the dataset folders and returns a dictionary
        mapping each class label to a list of loaded grayscale images. The method
        also stores the corresponding file paths in self.image_paths.
        """
        images_dict = {category: [] for category in self.categories}
        paths_dict = {category: [] for category in self.categories}

        for folder in self.folders:
            for category in self.categories:
                category_path = os.path.join(folder, category)

                if not os.path.isdir(category_path):
                    continue

                for filename in sorted(os.listdir(category_path)):
                    if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
                        continue

                    file_path = os.path.join(category_path, filename)

                    img = tf.keras.utils.load_img(file_path, color_mode="grayscale")

                    images_dict[category].append(img)
                    paths_dict[category].append(file_path)

        self.image_paths = paths_dict
        return images_dict

    def summary(self):
        """
        Creates a summary table of the dataset. The table is a pandas DataFrame
        with two columns: label, n_images.
        """

        rows = []

        for category in self.categories:
            rows.append({
                "label": category,
                "n_images": len(self.images[category])
            })

        return pd.DataFrame(rows)

    def get_encoded_images(self, height, width, test_size=0.2, random_state=42,
                           return_paths=False):
        """
        Converts loaded images into NumPy arrays. Each image is resized to the
        requested shape, normalized to the [0, 1] range, and reshaped to (height, width, 1).
        Class labels are converted into one-hot encoded vectors.
        The functions then performs a stratified train/test split and returns
        the prepared training and test subsets. Optionally (if return_paths == True)
        the function also returns the file paths assigned to the training and test splits.
        """

        X = []
        y_indices = []
        image_paths = []

        for category, list_img in self.images.items():
            for i, img in enumerate(list_img):
                x = np.array(
                    img.resize((width, height)),
                    dtype=np.float32
                ) / 255.0

                X.append(x)
                y_indices.append(self.categories.index(category))
                image_paths.append(self.image_paths[category][i])

        X = np.array(X).reshape(-1, height, width, 1)
        y_indices = np.array(y_indices)
        Y = tf.keras.utils.to_categorical(y_indices, num_classes=len(self.categories))

        if return_paths:
            X_train, X_test, Y_train, Y_test, train_paths, test_paths = train_test_split(
                X, Y, image_paths, test_size=test_size, random_state=random_state,
                shuffle=True, stratify=y_indices
            )

            return X_train, X_test, Y_train, Y_test, train_paths, test_paths

        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y, est_size=test_size, random_state=random_state, shuffle=True,
            stratify=y_indices
        )

        return X_train, X_test, Y_train, Y_test

    def get_all_image_paths(self):
        all_paths = []

        for category in self.categories:
            all_paths.extend(self.image_paths[category])

        return all_paths

    def sample_image_paths(self, n, source_paths=None, random_state=None):
        """
        Randomly samples n image paths (from a list of paths  to sample
        from - optionally. If not provided, samples from the whole dataset). The
        value of n should not be larger than the number of all available images.
        Returns a list of sampled image paths.
        """

        if source_paths is None:
            source_paths = self.get_all_image_paths()

        rng = random.Random(random_state)
        return rng.sample(source_paths, n)

    def show_image_sequence(self, image_paths, n_cols=5, show_paths=False):
        """
        Displays a given sequence of image paths in the provided order.
        """

        n = len(image_paths)

        n_cols = min(n_cols, n)
        n_rows = int(np.ceil(n / n_cols))

        plt.figure(figsize=(3 * n_cols, 3 * n_rows))

        for i, path in enumerate(image_paths):
            label = os.path.basename(os.path.dirname(path))
            filename = os.path.basename(path)

            img = tf.keras.utils.load_img(
                path,
                color_mode="grayscale"
            )

            ax = plt.subplot(n_rows, n_cols, i + 1)
            ax.imshow(img, cmap="gray")

            if show_paths:
                ax.set_title(f"{label}\n{filename}", fontsize=8)
            else:
                ax.set_title(f"Label: {label}")

            ax.axis("off")

        plt.tight_layout()
        plt.show()

    def show_images(self, n=9, source_paths=None, random_state=None, n_cols=5):
          """
          Randomly selects n images [from a provided list of paths - optionally.
          If the list is not provided (source_paths is None), images are sampled
          from the whole dataset] and displays them.
          """
          sampled_paths = self.sample_image_paths(
              n=n,
              source_paths=source_paths,
              random_state=random_state
          )

          self.show_image_sequence(
              image_paths=sampled_paths,
              n_cols=n_cols,
              show_paths=False
          )

          return sampled_paths

### **Path to Dataset (Google Drive)**
  
---

In [ ]:
# Note:
"""
(1) Run the two lines under "Mounting the drive" only if you are using Google Colab
and the dataset is stored on Google Drive.
(2) Change the "PROJECT_DIR" and "CHECKPOINT_DIR" paths before running the notebook.
"""

# Mounting the drive
from google.colab import drive
drive.mount('/content/drive/')

# Separating the directory with processed images
PROJECT_DIR = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER"
PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "Datasets",
    "ASL_Processed_Images",
    "asl_processed"
)

# Directories to load the final dataset from
PROCESSED_TRAIN_DIR = os.path.join(PROCESSED_DIR, "train")
PROCESSED_TEST_DIR = os.path.join(PROCESSED_DIR, "test")

# Additional directories (not connected to loading the ASL dataset): saving models (checkpoints)
CHECKPOINT_DIR = "/content/drive/MyDrive/YOUR_CHECKPOINTS_FOLDER"

### **Loading the dataset**
  
---

In [ ]:
# Loading the dataset with a dedicated function
asl_dataset = Images_Dataset([
    PROCESSED_TRAIN_DIR,
    PROCESSED_TEST_DIR
])

### **Dataset preview**
  
---

In [ ]:
# Preview of a few random images
asl_dataset.show_images(n=4, random_state=SEED)

# **Model training and evaluation**

This part contains the code (classes, functions) needed to train and test models, as well as to perform the configuration search. The training and testing themselves are conducted in ***Main*** part.

### **Model (CNN) classes**
  
---

In [ ]:
class CNN_Model(tf.keras.Model):
    """
    * CNN model for multi-class image classification.
    * Architecture:
        Conv2D -> MaxPooling2D -> Conv2D -> MaxPooling2D -> ...
        -> Flatten -> Dense -> Dropout -> Dense(softmax)
    * Parameters:
      - categories: Number of output classes for classification.
      - layers: List of (Conv2D_config, MaxPool2D_config) pairs.
        Example:
        [
            ({"filters": 16, "kernel_size": (3,3)},
             {"pool_size": (2,2)}),

            ({"filters": 32, "kernel_size": (3,3)},
             {"pool_size": (2,2)})
        ]
    """

    def __init__(self, categories, layers, dense_units=128, dropout_rate=0.3):
        super().__init__()

        self.feature_extractor = tf.keras.Sequential([
            layer
            for args_conv, args_pool in layers
            for layer in (tf.keras.layers.Conv2D(**args_conv),
                          tf.keras.layers.MaxPool2D(**args_pool))
        ])

        self.flatten = tf.keras.layers.Flatten()
        self.classifier_head = tf.keras.Sequential([
            tf.keras.layers.Dense(dense_units, activation="relu"),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(categories, activation="softmax")
        ])

    def call(self, x, training=False):
        x = self.feature_extractor(x, training=training)
        x = self.flatten(x)
        return self.classifier_head(x, training=training)

In [ ]:
class Convolutional_Neural_Network:
    def __init__(self, X, Y, layers, dense_units=128, dropout_rate=0.30):
        self.X = X
        self.Y = Y

        self.model = CNN_Model(
            categories=Y.shape[1],
            layers=layers,
            dense_units=dense_units,
            dropout_rate=dropout_rate
        )

    # train the model
    def train(self, LR, epochs, batch_size=32, momentum=0.0, optimizer="Adam",
              validation_split=None, callbacks=None, verbose=1):
      if optimizer == "Adam":
        opt = tf.keras.optimizers.Adam(learning_rate=LR)
      elif optimizer == "SGD":
        opt = tf.keras.optimizers.SGD(learning_rate=LR, momentum=momentum)

      self.model.compile(optimizer=opt, loss=tf.keras.losses.CategoricalCrossentropy(),
                         metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")])

      self.loss_acc = self.model.fit(self.X, self.Y, epochs=epochs, batch_size=batch_size,
                                     verbose=verbose, validation_split=validation_split,
                                     callbacks=callbacks, shuffle=True)
      return self.loss_acc

    def evaluate(self, X_test, Y_test):
        return self.model.evaluate(X_test, Y_test, verbose=0)

    # plot loss and accuracy history
    def plot_loss_accuracy(self):
        fig = plt.figure(figsize=(8, 4))
        fig.suptitle('Log Loss and Accuracy over Epochs')
        labels = ['Training', 'Validation']

        # get training history to plot
        loss = self.loss_acc.history.get('loss')
        val_loss = self.loss_acc.history.get('val_loss')
        accuracy = self.loss_acc.history.get('accuracy')
        val_accuracy = self.loss_acc.history.get('val_accuracy')

        # add_subplot(nrows, ncolumns, index)
        ax = fig.add_subplot(1, 2, 1)
        ax.plot(loss, label=labels[0])

        if val_loss:
            ax.plot(val_loss, label=labels[1])

        ax.grid(True)
        ax.set(xlabel='Epochs', title='Log Loss')
        ax.legend(loc='upper right')

        ax = fig.add_subplot(1, 2, 2)
        ax.plot(accuracy, label=labels[0])

        if val_accuracy:
            ax.plot(val_accuracy, label=labels[1])

        ax.grid(True)
        ax.set(xlabel='Epochs', title='Accuracy')
        ax.legend(loc='lower right')

        plt.tight_layout()
        plt.show()

### **Trainin and testing functions**

---

In [ ]:
def run_model(ID, layers, dataset, img_size=IMG_SIZE, learn_rate=LR, epochs=EPOCHS,
              batch_size=BATCH_SIZE, test_size=TEST_SIZE, validation_split=VAL_SPLIT,
              optimizer="Adam", val_metric="val_accuracy", vm_mode="max",
              random_state=SEED, verbose=1, dense_units=128, dropout_rate=0.3,
              print_results=False, checkpoint_dir=CHECKPOINT_DIR):
    """
    Trains and evaluates one model configuration.
    The function performs the full workflow for a single model run:
        1. creates a train/test split,
        2. builds a CNN with the selected architecture and hyperparameters,
        3. trains the model,
        4. saves the best model weights according to the validation metric,
        5. evaluates both the final-epoch model and the best checkpoint,
    The function returns the model, metrics, data splits, paths and training history.

    This function is used both during configuration search and during
    the final repeated evaluation of the selected model configuration.
    """

    os.makedirs(checkpoint_dir, exist_ok=True)
    best_model_weights = os.path.join(
        checkpoint_dir,
        f"tmp/best_model_{ID}.weights.h5"
        )

    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=best_model_weights,
        save_weights_only=True,
        monitor=val_metric,
        mode=vm_mode,
        save_best_only=True,
        verbose=0
    )

    height, width = img_size

    X_train, X_test, Y_train, Y_test, train_paths, test_paths = dataset.get_encoded_images(
        height,
        width,
        test_size=test_size,
        random_state=random_state,
        return_paths=True
    )

    m = Convolutional_Neural_Network(
        X_train,
        Y_train,
        layers=layers,
        dense_units=dense_units,
        dropout_rate=dropout_rate
    )

    history = m.train(
        LR=LR,
        optimizer=optimizer,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=[model_checkpoint],
        verbose=verbose
    )

    loss_last, acc_last = m.model.evaluate(X_test, Y_test, verbose=0)

    m.model.load_weights(best_model_weights)

    loss_best, acc_best = m.model.evaluate(X_test, Y_test, verbose=0)

    if print_results:
        print(f"Model {ID} layout:\n")

        for n, a in enumerate(layers, 1):
            print(
                f"Layer {n}: "
                f"{a[0]['filters']} filters, "
                f"kernel={a[0]['kernel_size']}, "
                f"pool={a[1]['pool_size']}"
            )

        print()

        print(
            f"Evaluation after {epochs} epochs:\n"
            f"Loss: {loss_last:.4f}, Accuracy: {acc_last:.4f}\n"
        )

        print(
            f"Best model evaluation:\n"
            f"Loss: {loss_best:.4f}, Accuracy: {acc_best:.4f}\n"
        )

    return {
        "ID": ID,
        "network": m,
        "X_train": X_train,
        "X_test": X_test,
        "Y_train": Y_train,
        "Y_test": Y_test,
        "history": history,
        "loss_last": loss_last,
        "acc_last": acc_last,
        "loss_best": loss_best,
        "acc_best": acc_best,
        "weights_path": best_model_weights,
        "train_paths": train_paths,
        "test_paths": test_paths
    }

In [ ]:
def average_performance(best_config, architectures, dataset,
                        runs=FINAL_RUNS, final_epochs=EPOCHS,
                        validation_split=VAL_SPLIT, optimizer="Adam",
                        val_metric="val_accuracy", vm_mode="max",
                        seeds=None, verbose=0):
    """
    Repeats the final model training a given number of times (the "runs" parameter)
    and reports performance stability. The function trains the selected best
    configuration multiple times using the same train/test split ratio but different
    random seeds (which can be passed in a form of a list to the "seeds" parameter.
    If such a list is not passed, a default seeds list - defined at the very beginning
    of this notebook, in "Reproducibility and configuration" chunk - is used).
    The functions returns a pandas DataFrame ("results_df") with one row per final
    model run and a dictionary ("summary") containing best, mean and standard
    deviation of accuracy.
    """

    if seeds is None:
        seeds = SEEDS

    if len(seeds) < runs:
        seeds = SEEDS

    architecture_name = best_config["architecture"]
    layers = architectures[architecture_name]

    accuracies = []
    rows = []

    for run_id in range(1, runs + 1):
        seed = seeds[run_id - 1]

        print(f"Final run {run_id}/{runs}, random_state={seed}")

        experiment = run_model(
            ID=f"final_{architecture_name}_run_{run_id}",
            layers=layers,
            dataset=dataset,
            img_size=best_config["img_size"],
            learn_rate=best_config["learning_rate"],
            epochs=final_epochs,
            batch_size=best_config["batch_size"],
            test_size=best_config["test_size"],
            validation_split=validation_split,
            optimizer=optimizer,
            val_metric=val_metric,
            vm_mode=vm_mode,
            random_state=seed,
            verbose=verbose,
            dense_units=best_config["dense_units"],
            dropout_rate=best_config["dropout_rate"],
            print_results=False
        )

        acc = experiment["acc_best"]
        loss = experiment["loss_best"]

        accuracies.append(acc)

        rows.append({
            "run": run_id,
            "random_state": seed,
            "test_size": best_config["test_size"],
            "loss_best": loss,
            "accuracy_best": acc,
            "weights_path": experiment["weights_path"]
        })

        print(f"Accuracy: {acc:.4f}\n")

    results_df = pd.DataFrame(rows)

    summary = {
        "runs": runs,
        "test_size": best_config["test_size"],
        "best_accuracy": float(np.max(accuracies)),
        "mean_accuracy": float(np.mean(accuracies)),
        "std_accuracy": float(np.std(accuracies))
    }

    display(
        results_df.style.hide(axis="index").set_caption("FINAL MODEL RUNS")
    )

    print("Final model performance:")
    print(f"Test size: {summary['test_size']}")
    print(f"Best accuracy: {summary['best_accuracy']:.4f}")
    print(f"Mean accuracy: {summary['mean_accuracy']:.4f}")
    print(f"STD accuracy: {summary['std_accuracy']:.4f}")

    return results_df, summary

### **Configuration search function**
  
---

In [ ]:
# Helper functions (for configuration search)
def sample_configurations(hyperparameters, n_configs=10, random_state=42):
    """
    Randomly samples hyperparameter configurations without duplicates and returns
    them in a form of a list. The function requires a dictionary of (lists of)
    hyperparameter configurations
    """

    keys = list(hyperparameters.keys())
    values = [hyperparameters[key] for key in keys]

    all_configs = [
        dict(zip(keys, combination))
        for combination in product(*values)
    ]

    rng = random.Random(random_state)

    if n_configs >= len(all_configs):
        return all_configs

    return rng.sample(all_configs, n_configs)

def get_best_history_value(history, metric, mode="max"):
    values = history.history.get(metric)

    if mode == "max":
        return float(np.max(values))

    if mode == "min":
        return float(np.min(values))

# The configuration search function
def config_search(hyperparameters, architectures, dataset,
                  n_configs=10, search_epochs=SEARCH_EPOCHS,
                  validation_split=VAL_SPLIT, optimizer="Adam",
                  val_metric="val_accuracy", vm_mode="max",
                  random_state=SEED, verbose=0):
    """
    Performs random configuration search for the CNN classifier. Returns a pandas
    DataFrame containing tested configurations, validation scores and test-set
    performance for each configuration.
    """

    configs = sample_configurations(
        hyperparameters=hyperparameters,
        n_configs=n_configs,
        random_state=random_state
    )

    results = []

    for n, config in enumerate(configs, 1):
        print(f"Running configuration {n}/{len(configs)}:")
        print(config)

        architecture_name = config["architecture"]
        layers = architectures[architecture_name]

        experiment = run_model(
            ID=f"search_config_{n}",
            layers=layers,
            dataset=dataset,
            img_size=config["img_size"],
            learn_rate=config["learning_rate"],
            epochs=search_epochs,
            batch_size=config["batch_size"],
            test_size=config["test_size"],
            validation_split=validation_split,
            optimizer=optimizer,
            val_metric=val_metric,
            vm_mode=vm_mode,
            random_state=random_state,
            verbose=verbose,
            dense_units=config["dense_units"],
            dropout_rate=config["dropout_rate"],
            print_results=False
        )

        selection_score = get_best_history_value(
            history=experiment["history"],
            metric=val_metric,
            mode=vm_mode
        )

        row = {
            "config": n,
            "architecture": architecture_name,
            "learning_rate": config["learning_rate"],
            "batch_size": config["batch_size"],
            "dense_units": config["dense_units"],
            "dropout_rate": config["dropout_rate"],
            "img_size": config["img_size"],
            "test_size": config["test_size"],
            "selection_metric": val_metric,
            "selection_score": selection_score,
            "test_accuracy_best": experiment["acc_best"],
            "test_loss_best": experiment["loss_best"]
        }

        results.append(row)

        print(
            f"Best {val_metric}: {selection_score:.4f}; "
            f"test accuracy: {experiment['acc_best']:.4f}\n"
        )

    df = pd.DataFrame(results)

    ascending = False if vm_mode == "max" else True

    df = df.sort_values(
        by="selection_score",
        ascending=ascending
    ).reset_index(drop=True)

    display(
        df.style.hide(axis="index").set_caption("CONFIGURATION SEARCH RESULTS")
    )

    return df

### **Breakdown table function**
---

In [ ]:
def make_breakdown_table(model, Y_train, X_test, Y_test, categories):
    """
    Creates and returns a table (pandas DataFrame) which summarizes the model's
    perfromance per class.
    """
    y_train = np.argmax(Y_train, axis=1)
    y_test = np.argmax(Y_test, axis=1)

    probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)

    rows = []

    for class_id, label in enumerate(categories):
        train_count = int(np.sum(y_train == class_id))
        test_count = int(np.sum(y_test == class_id))

        correct_predictions = int(
            np.sum((y_test == class_id) & (y_pred == class_id))
        )

        total_count = train_count + test_count

        if test_count > 0:
            accuracy = correct_predictions / test_count
        else:
            accuracy = np.nan

        rows.append({
            "Label": label,
            "Total data": total_count,
            "Training data": train_count,
            "Test data": test_count,
            "Correct predictions": correct_predictions,
            "Accuracy": accuracy
        })

    breakdown_df = pd.DataFrame(rows)

    return breakdown_df

# **Main**

### **Model architecture (layers)**

---

In [ ]:
cnn_architectures = {
    "small": [
        (
            {
                "filters": 16,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        ),
        (
            {
                "filters": 32,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        )
    ],

    "medium": [
        (
            {
                "filters": 16,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        ),
        (
            {
                "filters": 32,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        ),
        (
            {
                "filters": 64,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        )
    ],

    "large": [
        (
            {
                "filters": 32,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        ),
        (
            {
                "filters": 64,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        ),
        (
            {
                "filters": 128,
                "kernel_size": (3, 3),
                "activation": "relu",
                "padding": "same"
            },
            {
                "pool_size": (2, 2)
            }
        )
    ]
}

cnn_hyperparameters_dict = {
    "architecture": ["small", "medium", "large"],
    "learning_rate": [LR, 0.005, 0.0001],
    "batch_size": [16, BATCH_SIZE, 64],
    "dense_units": [64, 128, 256],
    "dropout_rate": [0.20, 0.30, 0.40],
    "img_size": [IMG_SIZE],
    "test_size": TEST_SPLITS
}

### **Train, test and evaluate models**
  
---

In [ ]:
# Performing the configuration search
search_results = config_search(
    hyperparameters=cnn_hyperparameters_dict,
    architectures=cnn_architectures,
    dataset=asl_dataset,
    n_configs=8
)

# Selecting the best hyperparameter configuration
best_config_row = search_results.sort_values(by="selection_score",
                                             ascending=False).iloc[0]

best_config = {
    "architecture": best_config_row["architecture"],
    "learning_rate": float(best_config_row["learning_rate"]),
    "batch_size": int(best_config_row["batch_size"]),
    "dense_units": int(best_config_row["dense_units"]),
    "dropout_rate": float(best_config_row["dropout_rate"]),
    "img_size": best_config_row["img_size"],
    "test_size": float(best_config_row["test_size"])
}

In [ ]:
# Training and evaluating the selected final configuration over multiple random seeds
# (five runs)
final_results, final_summary = average_performance(
    best_config=best_config,
    architectures=cnn_architectures,
    dataset=asl_dataset
)

### **Final model**  

---

#### **Saving the best run**

In [ ]:
# Selecting the best final run as a final model (to be used for the breakdown
# table and gesture translation later)
best_run_row = final_results.sort_values(
    by="accuracy_best",
    ascending=False
).iloc[0]

best_weights_path = best_run_row["weights_path"]
best_run_seed = int(best_run_row["random_state"])

print("Best final run:")
display(best_run_row)

print("Best model weights:", best_weights_path)
print("Best run seed:", best_run_seed)

In [ ]:
# Recreating the best final run (in order to obtain loss and accuracy - to plot
# them later)
best_run_for_plots = run_model(
    ID="best_final_run_plots",
    layers=cnn_architectures[best_config["architecture"]],
    dataset=asl_dataset,
    img_size=best_config["img_size"],
    learn_rate=best_config["learning_rate"],
    epochs=EPOCHS,
    batch_size=best_config["batch_size"],
    test_size=best_config["test_size"],
    validation_split=VAL_SPLIT,
    optimizer="Adam",
    val_metric="val_accuracy",
    vm_mode="max",
    random_state=best_run_seed,
    verbose=1,
    dense_units=best_config["dense_units"],
    dropout_rate=best_config["dropout_rate"],
    print_results=True
)

In [ ]:
# Recreating the best final run's split (for the breakdown table and sign language translation)
X_train_best, X_test_best, Y_train_best, Y_test_best, train_paths_best, test_paths_best = asl_dataset.get_encoded_images(
    height=best_config["img_size"][0],
    width=best_config["img_size"][1],
    test_size=best_config["test_size"],
    random_state=best_run_seed,
    return_paths=True
)

#### **Loading final model**
  
---

In [ ]:
# The function to load the selected final model
def load_model_from_best_config(best_config, architectures, n_classes, weights_path):
    architecture_name = best_config["architecture"]
    layers = architectures[architecture_name]

    height, width = best_config["img_size"]

    example_X = np.zeros((1, height, width, 1), dtype=np.float32)
    example_Y = np.zeros((1, n_classes), dtype=np.float32)

    m = Convolutional_Neural_Network(
        example_X,
        example_Y,
        layers=layers,
        dense_units=best_config["dense_units"],
        dropout_rate=best_config["dropout_rate"]
    )

    # This line only needs to be executed, but we will not use its outcome later -
    # hence the " _ " name
    _ = m.model(tf.zeros((1, height, width, 1)))

    m.model.load_weights(weights_path)

    return m.model

In [ ]:
# Loading the final (best) model
final_model = load_model_from_best_config(
    best_config=best_config,
    architectures=cnn_architectures,
    n_classes=len(asl_dataset.categories),
    weights_path=best_weights_path
)

### **Results**

#### **Configuration search results**
---

In [ ]:
print("Best configuration:")
display(best_config)

#### **Breakdown table**
  
---

In [ ]:
# The breakdown table for the final model (best run)
breakdown_table = make_breakdown_table(
    model=final_model,
    Y_train=Y_train_best,
    X_test=X_test_best,
    Y_test=Y_test_best,
    categories=asl_dataset.categories
)

display(
    breakdown_table.style
    .format({"Accuracy": "{:.4f}"})
    .hide(axis="index")
    .set_caption("BREAKDOWN TABLE BY CLASS")
)

#### **Plots**  

---

In [ ]:
# Plotting the loss and accuracy for the best final run
best_run_for_plots["network"].plot_loss_accuracy()

### **Sign language translation**

#### **Processing images - functions**
  
---

In [ ]:
def preprocess_single_image(image_source, img_size):
    """
    Loads a single image from a file path and prepares it for the CNN model.
    The image is loaded in grayscale, resized to the target size, normalized
    to the [0, 1] range, and reshaped to (1, height, width, 1).
    """

    height, width = img_size

    img = tf.keras.utils.load_img(
            image_source,
            color_mode="grayscale",
            target_size=(height, width)
        )

    arr = tf.keras.utils.img_to_array(img)
    arr = arr / 255.0

    return arr.reshape(1, height, width, 1).astype("float32")

In [ ]:
def predict_single_sign(model, image_source, categories, img_size):
    """
    Predicts the class label of a single sign image (hand gesture).
    """

    x = preprocess_single_image(image_source, img_size)

    probs = model.predict(x, verbose=0)[0] # Predicted probabilities
    class_id = int(np.argmax(probs)) # Predicted class index

    predicted_label = categories[class_id]
    predicted_class_prob = float(probs[class_id])

    return predicted_label, predicted_class_prob

In [ ]:
def translate_gesture_sequence(model, image_sequence, categories, img_size):
    """
    Translates a sequence of independently classified gesture images
    into a text string. Each image is classified separately.
    """

    predicted_chars = []
    rows = []

    for position, image_source in enumerate(image_sequence, start=1):
        label, confidence = predict_single_sign(
            model=model,
            image_source=image_source,
            categories=categories,
            img_size=img_size
        )

        predicted_chars.append(str(label))

        rows.append({
            "position": position,
            "predicted_label": label,
            "predicted_class_probabiblity": confidence
        })

    predicted_text = "".join(predicted_chars)
    prediction_table = pd.DataFrame(rows)

    return predicted_text, prediction_table

#### **Performing the translation**

In this part the model's ability to translate ASL signs into text is demonstarted. Three sets of examples are prepared: one constisting of three manually selected ASL signs that represent letters, one consiting of four manually selected ASL signs representing numbers and one with five randomly drawn images.

In [ ]:
# Preparing the first set of images
example01_labels = ["A", "S", "L"]
example01_paths = []

for label in example01_labels:
    candidates = [
        path for path in test_paths_best
        if os.path.basename(os.path.dirname(path)) == label
    ]

    example01_paths.append(candidates[0])

# Printing the paths to selected images and showing them
print("Manually selected image paths:")
for path in example01_paths:
    print(path)

# Showing the chosen images
asl_dataset.show_image_sequence(example01_paths)

In [ ]:
# Example translation no. 1: three letter signs
predicted_text01, prediction_table01 = translate_gesture_sequence(
    model=final_model,
    image_sequence=example01_paths,
    categories=asl_dataset.categories,
    img_size=best_config["img_size"]
)

print("Expected text:", "".join(example01_labels))
print("Predicted text:", predicted_text01)

display(prediction_table01)

In [ ]:
# Preparing the second set of images
example02_labels = ["2", "0", "2", "6"]
example02_paths = []

for label in example02_labels:
    candidates = [
        path for path in test_paths_best
        if os.path.basename(os.path.dirname(path)) == label
    ]

    example02_paths.append(candidates[0])

# Printing the paths to selected images and showing them
print("Manually selected image paths:")
for path in example02_paths:
    print(path)

# Showing the chosen images
asl_dataset.show_image_sequence(example02_paths)

In [ ]:
# Example translation no. 2: four number signs
predicted_text02, prediction_table02 = translate_gesture_sequence(
    model=final_model,
    image_sequence=example02_paths,
    categories=asl_dataset.categories,
    img_size=best_config["img_size"]
)

print("Expected text:", "".join(example02_labels))
print("Predicted text:", predicted_text02)

display(prediction_table02)

In [ ]:
# Preparing the third set of images
rng = random.Random(SEED)

# Drawing the paths first
example03_paths = rng.sample(test_paths_best, k=5)

# Showing the labels
example03_labels = "".join(
    os.path.basename(os.path.dirname(path))
    for path in example03_paths
)

# Printing the paths to selected images and showing them
print("Randomly selected image paths:")
for path in example03_paths:
    print(path)

# Showing the chosen images
asl_dataset.show_image_sequence(example03_paths)

In [ ]:
# Example translation no. 3: five random images
predicted_text03, prediction_table03 = translate_gesture_sequence(
    model=final_model,
    image_sequence=example03_paths,
    categories=asl_dataset.categories,
    img_size=best_config["img_size"]
)

print("Expected text:", "".join(example03_labels))
print("Predicted text:", predicted_text03)

display(prediction_table03)